# Human vs 3 AI (Interactive UI)

Use the dropdown/button to choose your action for `HUMAN_SEAT`.
The other 3 seats use `RandomAgent`.


In [ ]:
import random
import json

from IPython.display import display, clear_output
import ipywidgets as widgets

from riichienv import RiichiEnv
from riichienv.agents import RandomAgent
from riichienv.visualizer import GameViewer


In [ ]:
HUMAN_SEAT = 0
GAME_MODE = "4p-red-half"

ai_agent = RandomAgent()

state = {
    "env": None,
    "obs_dict": None,
    "human_obs": None,
    "pending_ai_actions": {},
}

ACTION_TYPE_CN = {
    "dahai": "打牌",
    "chi": "吃",
    "pon": "碰",
    "daiminkan": "大明杠",
    "ankan": "暗杠",
    "kakan": "加杠",
    "reach": "立直",
    "hora": "和牌",
    "ryukyoku": "流局",
    "none": "过",
    "nukidora": "拔北",
}

action_dropdown = widgets.Dropdown(description="动作", options=[], layout=widgets.Layout(width="95%"))
play_button = widgets.Button(description="出牌", button_style="success")
random_button = widgets.Button(description="随机")
reset_button = widgets.Button(description="重开", button_style="warning")

status_out = widgets.Output()
viewer_out = widgets.Output()
ui = widgets.VBox([widgets.HBox([play_button, random_button, reset_button]), action_dropdown, status_out, viewer_out])


def _format_action(action):
    try:
        ev = json.loads(action.to_mjai())
        t = ev.get("type", "")
        label = ACTION_TYPE_CN.get(t, t or "未知动作")

        parts = [label]
        pai = ev.get("pai")
        if pai is not None:
            parts.append(str(pai))

        consumed = ev.get("consumed")
        if consumed:
            parts.append("(" + ",".join(str(x) for x in consumed) + ")")

        target = ev.get("target")
        if target is not None:
            parts.append(f"<-{target}")

        return " ".join(parts)
    except Exception:
        return str(action)


def _set_human_options(obs):
    legal = obs.legal_actions()
    opts = [(f"[{i}] {_format_action(a)}", i) for i, a in enumerate(legal)]
    action_dropdown.options = opts
    action_dropdown.value = 0 if opts else None


def _render():
    env = state["env"]
    with status_out:
        clear_output(wait=True)
        print("done:", env.done())
        print("scores:", list(env.scores()))
        print("ranks:", list(env.ranks()))
        if env.done():
            print("Game finished.")
        elif state["human_obs"] is None:
            print("Waiting for next human turn...")
        else:
            print("Your turn.")

    with viewer_out:
        clear_output(wait=True)
        current_step = max(len(getattr(env, "mjai_log", [])) - 1, 0)
        display(GameViewer.from_env(env, step=current_step, perspective=HUMAN_SEAT, freeze=True))


def _advance_until_human_turn():
    env = state["env"]
    obs_dict = state["obs_dict"]

    while not env.done():
        ai_actions = {}
        human_obs = None

        for pid, obs in obs_dict.items():
            legal = obs.legal_actions()
            if not legal:
                continue
            if pid == HUMAN_SEAT:
                human_obs = obs
            else:
                ai_actions[pid] = ai_agent.act(obs)

        if human_obs is not None:
            state["human_obs"] = human_obs
            state["pending_ai_actions"] = ai_actions
            state["obs_dict"] = obs_dict
            _set_human_options(human_obs)
            return

        if not ai_actions:
            break

        obs_dict = env.step(ai_actions)

    state["human_obs"] = None
    state["pending_ai_actions"] = {}
    state["obs_dict"] = obs_dict
    action_dropdown.options = []


def reset_game(_=None):
    env = RiichiEnv(game_mode=GAME_MODE)
    obs_dict = env.reset()
    state["env"] = env
    state["obs_dict"] = obs_dict
    state["human_obs"] = None
    state["pending_ai_actions"] = {}
    _advance_until_human_turn()
    _render()


def play_selected(_):
    env = state["env"]
    obs = state["human_obs"]
    if env is None or obs is None or env.done():
        _render()
        return

    legal = obs.legal_actions()
    idx = action_dropdown.value
    if idx is None or idx < 0 or idx >= len(legal):
        return

    actions = dict(state["pending_ai_actions"])
    actions[HUMAN_SEAT] = legal[idx]
    state["obs_dict"] = env.step(actions)
    state["human_obs"] = None
    state["pending_ai_actions"] = {}
    _advance_until_human_turn()
    _render()


def play_random(_):
    if not action_dropdown.options:
        return
    action_dropdown.value = random.choice([v for _, v in action_dropdown.options])
    play_selected(None)


play_button.on_click(play_selected)
random_button.on_click(play_random)
reset_button.on_click(reset_game)

display(ui)
reset_game()

